# 02 · Cleaning + leakage audit

**Question:** which columns can the model honestly use when a lead first arrives?
A counsellor ranking tomorrow's calls only knows what the lead told us at sign-up (plus web activity so far).
Anything filled in *after* a counsellor talks to the lead (call notes, quality ratings) would make the model
look brilliant in testing and useless in practice.

All cleaning rules are learned on the **training split only** (`LeadCleaner.fit`) and then applied unchanged
to the test set. Logic: `src/clean.py`, `src/metrics.py`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import roc_auc_score

from src.config import FIG_DIR, ROOT as REPO, load_config
from src.clean import load_raw, LeadCleaner, leakage_audit, ID_COLS, LEAKY_COLS, TARGET
from src.model import split, logistic
from src import metrics, eda

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)
cfg = load_config()
RS = cfg["model"]["random_state"]

## 1. Split first, then clean

In [2]:
df = load_raw()
train, test = split(df, cfg)
print(f"train {len(train):,} leads (conv {train[TARGET].mean():.1%}) | "
      f"test {len(test):,} leads (conv {test[TARGET].mean():.1%})  — stratified, random_state={RS}")

train 6,930 leads (conv 38.5%) | test 2,310 leads (conv 38.5%)  — stratified, random_state=42


## 2. Fit the cleaner on training data

In [3]:
cleaner = LeadCleaner().fit(train)
X_train, X_test = cleaner.transform(train), cleaner.transform(test)
y_train, y_test = train[TARGET], test[TARGET]
print(f"{X_train.shape[1]} model features from {df.shape[1] - 1} raw columns")
display(cleaner.decisions())

54 model features from 36 raw columns


,action,reason
column,,
Prospect ID,dropped,identifier
Lead Number,dropped,identifier
Tags,dropped,post-contact (leakage)
Lead Quality,dropped,post-contact (leakage)
Lead Profile,dropped,post-contact (leakage)
Asymmetrique Activity Index,dropped,post-contact (leakage)
Asymmetrique Profile Index,dropped,post-contact (leakage)
Asymmetrique Activity Score,dropped,post-contact (leakage)
Asymmetrique Profile Score,dropped,post-contact (leakage)


## 3. Leakage audit

`when_known`: **creation** = known when the lead arrives · **post-contact** = filled in after a counsellor
engages · **unclear** = could be either; flagged for the project owner and kept unless told otherwise.

In [4]:
audit = leakage_audit(cleaner, list(df.columns))
display(audit)
(REPO / "reports").mkdir(exist_ok=True)
audit.to_csv(REPO / "reports" / "leakage_audit.csv", index=False)

def to_md(t):
    head = "| " + " | ".join(t.columns) + " |\n|" + "---|" * len(t.columns) + "\n"
    return head + "\n".join("| " + " | ".join(str(v).replace("|", "/") for v in r) + " |"
                             for r in t.itertuples(index=False))

(REPO / "reports" / "leakage_audit.md").write_text(
    "# Leakage audit\n\nGenerated by notebooks/02_cleaning_leakage_audit.ipynb.\n\n" + to_md(audit) + "\n",
    encoding="utf-8")
print(audit.when_known.value_counts().to_string())

,column,when_known,decision,reason
0,Prospect ID,id,dropped,"Identifier, no predictive meaning"
1,Lead Number,id,dropped,"Identifier, no predictive meaning"
2,Lead Origin,creation,kept (one-hot),"How the lead entered the system (form, API, import). 3 levels kept, rest → Other; blanks → 'Missing' level"
3,Lead Source,creation,kept (one-hot),"Traffic source that produced the lead. 7 levels kept, rest → Other; blanks → 'Missing' level"
4,Do Not Email,unclear,kept (Yes/No → 1/0),Opt-out flag; could be set after the lead starts receiving emails
5,Do Not Call,unclear,dropped,"Opt-out flag; could be set after a call. Kept for training, excluded from call lists. near-constant (100.0% one value)"
6,TotalVisits,unclear,kept (numeric),Web activity measured at extraction; may include visits after first contact. capped at 99th pct = 16; median-imputed...
7,Total Time Spent on Website,unclear,kept (numeric),Web activity measured at extraction; may include visits after first contact. capped at 99th pct = 1834.42
8,Page Views Per Visit,unclear,kept (numeric),Web activity measured at extraction; may include visits after first contact. capped at 99th pct = 9; median-imputed ...
9,Last Activity,post-contact,dropped,"Latest logged activity, includes counsellor actions (e.g. 'SMS Sent', 'Had a Phone Conversation')"


when_known
creation        18
post-contact     9
unclear          7
id               2

### Columns flagged as *unclear* (kept for now)

In [5]:
display(audit[audit.when_known == "unclear"])

,column,when_known,decision,reason
4,Do Not Email,unclear,kept (Yes/No → 1/0),Opt-out flag; could be set after the lead starts receiving emails
5,Do Not Call,unclear,dropped,"Opt-out flag; could be set after a call. Kept for training, excluded from call lists. near-constant (100.0% one value)"
6,TotalVisits,unclear,kept (numeric),Web activity measured at extraction; may include visits after first contact. capped at 99th pct = 16; median-imputed...
7,Total Time Spent on Website,unclear,kept (numeric),Web activity measured at extraction; may include visits after first contact. capped at 99th pct = 1834.42
8,Page Views Per Visit,unclear,kept (numeric),Web activity measured at extraction; may include visits after first contact. capped at 99th pct = 9; median-imputed ...
13,What is your current occupation,unclear,kept (one-hot),"Form field, but blank together with 'What matters most' for 99% of blanks: may be a later form step or counsellor-fi..."
14,What matters most to you in choosing a course,unclear,kept (one-hot),"Form field, but blank together with occupation for 99% of blanks: may be a later form step or counsellor-filled. 2 l..."


## 4. Leakage proof: same model, with and without the post-contact columns

Identical pipeline and logistic regression; the only difference is whether `Tags`, `Lead Quality`,
`Lead Profile`, `Last Activity`, `Last Notable Activity` and the `Asymmetrique` scores are allowed in.
Both evaluated on the same held-out test leads.

In [6]:
def run(cl):
    Xtr, Xte = cl.fit_transform(train), cl.transform(test)
    m = logistic(random_state=RS).fit(Xtr, y_train)
    p = m.predict_proba(Xte)[:, 1]
    return m, Xtr.columns, p, {"Test AUC": roc_auc_score(y_test, p),
                               "Precision @ top 20%": metrics.precision_at_k(y_test, p, 0.2)}

m_clean, f_clean, p_clean, r_clean = run(LeadCleaner())
m_leaky, f_leaky, p_leaky, r_leaky = run(LeadCleaner(drop_cols=ID_COLS))

res = pd.DataFrame({"Honest (creation-time)": r_clean, "Leaky (+ post-contact)": r_leaky}).T
display(res.style.format({"Test AUC": "{:.3f}", "Precision @ top 20%": "{:.1%}"}))
res.to_csv(REPO / "reports" / "leakage_experiment.csv")

gap_auc = r_leaky["Test AUC"] - r_clean["Test AUC"]
title = (f"Post-contact columns inflate test AUC from {r_clean['Test AUC']:.2f} to {r_leaky['Test AUC']:.2f} "
         f"and precision@top-20% from {r_clean['Precision @ top 20%']:.0%} to "
         f"{r_leaky['Precision @ top 20%']:.0%} — numbers the model could never hit on a new lead")
eda.plot_leakage_gap(res, title, FIG_DIR / "02_leakage_gap.png")
print(title)

,Test AUC,Precision @ top 20%
Honest (creation-time),0.852,82.3%
Leaky (+ post-contact),0.976,98.5%


Post-contact columns inflate test AUC from 0.85 to 0.98 and precision@top-20% from 82% to 98% — numbers the model could never hit on a new lead


### Why the leaky model looks so good: its strongest features are counsellor notes

In [7]:
coef = pd.Series(m_leaky.named_steps["lr"].coef_[0], index=f_leaky)
top = coef.abs().sort_values(ascending=False).head(10).index
leaky_top = pd.DataFrame({"coef (scaled)": coef[top],
                          "post-contact?": [any(f.startswith(c) for c in LEAKY_COLS) for f in top]})
display(leaky_top)
print(f"{leaky_top['post-contact?'].sum()} of the leaky model's top 10 features are post-contact columns.")

,coef (scaled),post-contact?
Tags=Ringing,-1.453550,True
Tags=Will revert after reading the email,1.285950,True
Total Time Spent on Website,1.137909,False
Tags=Closed by Horizzon,1.108664,True
Tags=Lost to EINS,1.010255,True
Tags=switched off,-0.860225,True
Tags=Interested in other courses,-0.730519,True
Tags=Already a student,-0.708732,True
Tags=Missing,0.599833,True
Lead Quality=Worst,-0.584439,True


9 of the leaky model's top 10 features are post-contact columns.


## 4b. How much does the honest model lean on the *unclear* columns?

If any unclear group later turns out to be post-contact, this is what we'd lose. Each row drops one group
from the honest model (everything else identical).

In [8]:
UNCLEAR_GROUPS = {
    "occupation + what matters most": ["What is your current occupation",
                                       "What matters most to you in choosing a course"],
    "web activity (visits, time, page views)": ["TotalVisits", "Total Time Spent on Website",
                                                "Page Views Per Visit"],
    "Do Not Email": ["Do Not Email"],
}
rows = {"Honest model (all kept)": r_clean}
for name, cols in UNCLEAR_GROUPS.items():
    rows[f"without {name}"] = run(LeadCleaner(drop_cols=ID_COLS + LEAKY_COLS + cols))[3]
sens = pd.DataFrame(rows).T
sens["AUC change"] = sens["Test AUC"] - r_clean["Test AUC"]
display(sens.style.format({"Test AUC": "{:.3f}", "Precision @ top 20%": "{:.1%}", "AUC change": "{:+.3f}"}))
sens.to_csv(REPO / "reports" / "unclear_columns_sensitivity.csv")

,Test AUC,Precision @ top 20%,AUC change
Honest model (all kept),0.852,82.3%,+0.000
without occupation + what matters most,0.817,77.7%,-0.035
"without web activity (visits, time, page views)",0.767,77.5%,-0.085
without Do Not Email,0.850,82.0%,-0.002


## 5. Do Not Call

In [9]:
dnc = (df["Do Not Call"] == "Yes")
print(f"Do Not Call = Yes: {dnc.sum()} of {len(df):,} leads. Kept in training data; "
      f"the column itself is near-constant so the cleaner drops it as a feature. "
      f"These leads are removed from every call list in the app.")

Do Not Call = Yes: 2 of 9,240 leads. Kept in training data; the column itself is near-constant so the cleaner drops it as a feature. These leads are removed from every call list in the app.
